# RAG v2 — MAX SCORE Edition

**What changed vs v1 (0.59)**

| Component | v1 | v2 |
|---|---|---|
| LLM | Qwen2.5-7B bf16 | **Qwen2.5-14B 4-bit** (2× params) |
| KB | 2K train rows only | **Train + Simple Wikipedia (100K articles)** |
| Docs per row | 3 | **4** (+ per-choice statements) |
| Reasoning | Single pass | **Self-consistency (2 seeds averaged)** |
| Prompt | Score directly | **Reason step-by-step → then score** |
| Query | 2 variants | **3 variants** (original + clean + choice-augmented) |

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
!pip install -q \
    "sentence-transformers==2.7.0" \
    "transformers>=4.44.0" \
    "huggingface_hub>=0.24.0" \
    accelerate bitsandbytes datasets \
    rank_bm25 faiss-gpu \
    tqdm pandas numpy \
  && pip uninstall -q -y torchcodec 2>/dev/null; echo DONE

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import json, re, gc, warnings, pickle
from pathlib import Path
from typing import List, Dict
from collections import defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import faiss
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    AutoModelForSequenceClassification, BitsAndBytesConfig,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
KAGGLE   = Path("/kaggle/input").exists()
DATA_DIR = Path("/kaggle/input") if KAGGLE else Path("Data")
OUT_DIR  = Path("/kaggle/working") if KAGGLE else Path("Submissions")
CACHE    = Path("/kaggle/working/cache") if KAGGLE else Path("rag_cache")
OUT_DIR.mkdir(exist_ok=True)
CACHE.mkdir(exist_ok=True)

TRAIN_PATH = next(DATA_DIR.rglob("train.csv"), DATA_DIR / "train.csv")
TEST_PATH  = next(DATA_DIR.rglob("test.csv"),  DATA_DIR / "test.csv")

# ── Models ────────────────────────────────────────────────────────────────────
EMBED_MODEL  = "BAAI/bge-base-en-v1.5"          # 0.4 GB  GPU-0
RERANK_MODEL = "BAAI/bge-reranker-large"         # 1.5 GB  GPU-0
LLM_MODEL    = "Qwen/Qwen2.5-14B-Instruct"      # ~8 GB 4-bit across GPU-0+1

# ── Retrieval ─────────────────────────────────────────────────────────────────
DENSE_K      = 40
SPARSE_K     = 40
RERANK_TOP_K = 6       # top-6 evidence pieces → richer context
ALPHA        = 0.65    # dense weight in RRF

# ── Wikipedia KB ──────────────────────────────────────────────────────────────
MAX_WIKI_ARTICLES = 100_000   # Simple Wikipedia ≈ 235K total; 100K is fast
CHUNK_SIZE        = 500       # tokens per chunk
CHUNK_OVERLAP     = 100

# ── LLM ──────────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS   = 300        # room for reasoning + JSON
SELF_CONSIST_N   = 2          # number of seeds to average
TEMPERATURES     = [0.05, 0.20]

CHOICES  = ["A", "B", "C", "D", "E"]
N_GPU    = torch.cuda.device_count()
DEVICE   = "cuda" if N_GPU > 0 else "cpu"
EMB_DEV  = "cuda:0"

print(f"GPUs : {N_GPU}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

In [ ]:
# ── Load Competition Data ─────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train {train_df.shape}   Test {test_df.shape}")

In [ ]:
# ── Build Knowledge Base ──────────────────────────────────────────────────────
# Part A: Training data  (immediate, no download)
# Part B: Simple Wikipedia (streamed from HuggingFace, ~100K articles)

KB_CACHE = CACHE / "kb_docs.pkl"

def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        end = min(start + size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words): break
        start = end - overlap
    return chunks


if KB_CACHE.exists():
    print("Loading cached KB...")
    with open(KB_CACHE, "rb") as f:
        kb_docs = pickle.load(f)
    print(f"✅ {len(kb_docs):,} docs loaded from cache")

else:
    kb_docs: List[str] = []

    # ── Part A: Training Q&A (4 doc types per row) ────────────────────────
    print("Building KB from training data...")
    for _, row in train_df.iterrows():
        correct_text = str(row[row["answer"]])
        all_choices  = " | ".join(f"{c}: {row[c]}" for c in CHOICES)

        # Doc 1 — Full solved Q&A
        kb_docs.append(
            f"Question: {row['prompt']} "
            f"Choices: [{all_choices}] "
            f"Correct: {row['answer']} — {correct_text}"
        )
        # Doc 2 — Correct answer as standalone fact
        kb_docs.append(f"Fact: {correct_text}")
        # Doc 3 — Question topic only
        kb_docs.append(f"Topic: {row['prompt']}")
        # Doc 4 — Each wrong answer labelled as wrong (negative knowledge)
        for c in CHOICES:
            if c != row["answer"]:
                kb_docs.append(f"Incorrect option for '{row['prompt'][:60]}...': {row[c]}")

    train_kb_size = len(kb_docs)
    print(f"  Training KB: {train_kb_size:,} docs")

    # ── Part B: Simple Wikipedia (100K articles, clean language) ──────────
    print(f"Streaming Simple Wikipedia ({MAX_WIKI_ARTICLES:,} articles)...")
    from datasets import load_dataset

    wiki = load_dataset(
        "wikimedia/wikipedia",
        "20231101.simple",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )

    wiki_chunk_count = 0
    for i, article in enumerate(tqdm(wiki, total=MAX_WIKI_ARTICLES, desc="Wikipedia")):
        if i >= MAX_WIKI_ARTICLES:
            break
        if len(article["text"].split()) < 30:
            continue
        for chunk in chunk_text(article["text"]):
            kb_docs.append(f"[{article['title']}] {chunk}")
            wiki_chunk_count += 1

    print(f"  Wikipedia chunks: {wiki_chunk_count:,}")
    print(f"  Total KB size   : {len(kb_docs):,} docs")

    with open(KB_CACHE, "wb") as f:
        pickle.dump(kb_docs, f)
    print(f"✅ KB cached → {KB_CACHE}")

In [ ]:
# ── BGE Embedding Model + FAISS Index ────────────────────────────────────────
FAISS_CACHE = CACHE / "faiss.index"
BGE_Q_PFX   = "Represent this sentence for searching relevant passages: "

print(f"Loading {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL, device=EMB_DEV)


def encode(texts: List[str], is_query=False, bs=512) -> np.ndarray:
    if is_query:
        texts = [BGE_Q_PFX + t for t in texts]
    return embedder.encode(
        texts, batch_size=bs, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True,
    ).astype(np.float32)


if FAISS_CACHE.exists():
    print("Loading cached FAISS index...")
    faiss_index = faiss.read_index(str(FAISS_CACHE))
    print(f"✅ FAISS: {faiss_index.ntotal:,} vectors")
else:
    print(f"Encoding {len(kb_docs):,} docs...")
    embs = encode(kb_docs, is_query=False, bs=512)
    dim  = embs.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(embs)
    faiss.write_index(faiss_index, str(FAISS_CACHE))
    print(f"✅ FAISS index built & cached: {faiss_index.ntotal:,} vectors")
    del embs; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── BM25 Index ────────────────────────────────────────────────────────────────
BM25_CACHE = CACHE / "bm25.pkl"

if BM25_CACHE.exists():
    print("Loading cached BM25...")
    with open(BM25_CACHE, "rb") as f:
        bm25 = pickle.load(f)
    print("✅ BM25 loaded")
else:
    print(f"Building BM25 over {len(kb_docs):,} docs...")
    bm25 = BM25Okapi([d.lower().split() for d in tqdm(kb_docs, desc="BM25 tokenize")])
    with open(BM25_CACHE, "wb") as f:
        pickle.dump(bm25, f)
    print("✅ BM25 built & cached")

In [ ]:
# ── BGE-reranker-large ────────────────────────────────────────────────────────
print(f"Loading {RERANK_MODEL}...")
rr_tok   = AutoTokenizer.from_pretrained(RERANK_MODEL)
rr_model = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL, torch_dtype=torch.float16
).to(EMB_DEV).eval()


@torch.no_grad()
def rerank(query: str, docs: List[str], top_k: int = RERANK_TOP_K) -> List[str]:
    if len(docs) <= top_k:
        return docs
    pairs  = [[query, d] for d in docs]
    enc    = rr_tok(pairs, padding=True, truncation=True,
                   max_length=512, return_tensors="pt").to(EMB_DEV)
    scores = rr_model(**enc).logits.squeeze(-1).float().cpu().numpy()
    ranked = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in ranked]


print(f"✅ Reranker ready on {EMB_DEV}")

# VRAM snapshot
for i in range(N_GPU):
    used = torch.cuda.memory_allocated(i)/1e9
    tot  = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f"  GPU {i}: {used:.1f}/{tot:.1f} GB")

In [ ]:
# ── Hybrid Retrieval ──────────────────────────────────────────────────────────
def rrf(dense: List[int], sparse: List[int], alpha=ALPHA, k=60) -> List[int]:
    s: Dict[int, float] = defaultdict(float)
    for r, i in enumerate(dense):  s[i] += alpha       * (1/(k+r+1))
    for r, i in enumerate(sparse): s[i] += (1-alpha)   * (1/(k+r+1))
    return sorted(s, key=s.__getitem__, reverse=True)


def dense_search(q: str, k: int = DENSE_K) -> List[int]:
    emb = encode([q], is_query=True, bs=1)
    _, I = faiss_index.search(emb, k)
    return I[0].tolist()


def sparse_search(q: str, k: int = SPARSE_K) -> List[int]:
    scores = bm25.get_scores(q.lower().split())
    return np.argsort(scores)[::-1][:k].tolist()


def retrieve(queries: List[str], top_k: int = RERANK_TOP_K) -> List[str]:
    """
    Multi-query hybrid retrieval:
      - Run dense + sparse for each query variant
      - RRF-fuse all results into one ranked list
      - Cross-encoder rerank → top_k
    """
    fused_scores: Dict[int, float] = defaultdict(float)
    for q in queries:
        dense  = dense_search(q)
        sparse = sparse_search(q)
        merged = rrf(dense, sparse)
        for rank, idx in enumerate(merged):
            fused_scores[idx] += 1.0 / (1 + rank)   # accumulate across queries

    pool_idxs = sorted(fused_scores, key=fused_scores.__getitem__, reverse=True)
    pool_docs = [kb_docs[i] for i in pool_idxs]

    return rerank(queries[0], pool_docs, top_k=top_k)


print("✅ Hybrid retrieval ready")

In [ ]:
# ── Load Qwen2.5-14B-Instruct (4-bit NF4 across both GPUs) ───────────────────
# 14B in 4-bit ≈ 8 GB  +  BGE models ≈ 2 GB  →  total ≈ 10 GB  (30 GB available)
print(f"Loading {LLM_MODEL} in 4-bit NF4 ...")

bnb = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.bfloat16,
)

llm_tok = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
llm     = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config = bnb,
    device_map          = "auto",
    torch_dtype         = torch.bfloat16,
    trust_remote_code   = True,
).eval()

print("VRAM after loading LLM:")
for i in range(N_GPU):
    u = torch.cuda.memory_allocated(i)/1e9
    t = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f"  GPU {i}: {u:.1f}/{t:.1f} GB")
print("✅ LLM ready")

In [ ]:
# ── Prompt Engineering ────────────────────────────────────────────────────────
# Two-step prompt: (1) explicit chain-of-thought reasoning, (2) JSON scores
# This consistently outperforms asking for scores directly.

SYSTEM = """You are an expert at solving multiple-choice questions across all subjects.

You will receive a question with 5 options (A–E) and retrieved knowledge evidence.

Your process:
STEP 1 — Read evidence carefully. Note what it supports or contradicts.
STEP 2 — Reason through each option briefly (1-2 sentences each).
STEP 3 — Output a JSON confidence score for each option.

Scoring rules:
  - Correct answer → score close to 1.0
  - Plausible but wrong → 0.1-0.4
  - Clearly wrong → close to 0.0

End your response with ONLY this JSON on the final line (no markdown fences):
{"A": 0.XX, "B": 0.XX, "C": 0.XX, "D": 0.XX, "E": 0.XX}"""


def build_prompt(row: pd.Series, evidence: List[str]) -> str:
    opts = "\n".join(f"  {c}. {row[c]}" for c in CHOICES)
    ev   = "\n\n".join(
        f"[Evidence {i+1}] {e[:350]}" for i, e in enumerate(evidence)
    )
    return (
        f"=== QUESTION ===\n{row['prompt']}\n\n"
        f"=== OPTIONS ===\n{opts}\n\n"
        f"=== RETRIEVED EVIDENCE ===\n{ev}\n\n"
        "Follow the 3-step process. End with JSON scores."
    )


def parse_scores(text: str) -> Dict[str, float]:
    """Extract final JSON score dict with multiple fallback strategies."""
    # Strategy 1: find the LAST {...} block (chain-of-thought puts JSON at end)
    blocks = re.findall(r'\{[^{}]+\}', text, re.DOTALL)
    for block in reversed(blocks):   # try last first
        try:
            s = json.loads(block)
            if len([c for c in CHOICES if c in s]) >= 3:
                return {c: float(s.get(c, 0.2)) for c in CHOICES}
        except Exception:
            pass

    # Strategy 2: regex per choice
    s = {}
    for c in CHOICES:
        m = re.search(rf'["\'{c}"]\s*:\s*([0-9]*\.?[0-9]+)', text)
        if m: s[c] = float(m.group(1))
    if len(s) >= 3:
        return {c: s.get(c, 0.0) for c in CHOICES}

    # Strategy 3: look for letter ranking in text ("B is correct", "answer: C")
    for pat in [r'\banswer(?:\s+is)?:?\s*([A-E])\b', r'\bcorrect(?:\s+option)?:?\s*([A-E])\b']:
        m = re.search(pat, text, re.I)
        if m:
            winner = m.group(1).upper()
            return {c: (0.8 if c == winner else 0.05) for c in CHOICES}

    return {c: 1.0 for c in CHOICES}   # last resort: uniform


print("✅ Prompt engine ready")

In [ ]:
# ── Query Expansion ───────────────────────────────────────────────────────────
FILLER = re.compile(
    r'^(pick|select|choose|determine|identify|find|state)\s+(the\s+)?(best|most|correct|right|accurate)'
    r'[\s\w]*?[:\s]+', re.I
)

def expand_queries(row: pd.Series) -> List[str]:
    """Generate 3 query variants for richer retrieval coverage."""
    prompt  = str(row["prompt"])
    clean   = FILLER.sub('', prompt).strip().rstrip('.,;:')

    # Augment with the first choice (usually a long descriptive option)
    best_choice = max(CHOICES, key=lambda c: len(str(row[c])))
    augmented   = f"{clean} {str(row[best_choice])[:120]}"

    queries = [prompt]
    if clean != prompt:
        queries.append(clean)
    queries.append(augmented)
    return queries[:3]


# Demo
sample_row = test_df.iloc[0]
for i, q in enumerate(expand_queries(sample_row), 1):
    print(f"Query {i}: {q[:100]}")

In [ ]:
# ── RAG Predict with Self-Consistency ─────────────────────────────────────────
@torch.no_grad()
def single_pass(user_msg: str, temperature: float, seed: int) -> Dict[str, float]:
    """One forward pass through Qwen → parsed score dict."""
    torch.manual_seed(seed)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": user_msg},
    ]
    text   = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok([text], return_tensors="pt", truncation=True, max_length=3500).to(DEVICE)

    out = llm.generate(
        **inputs,
        max_new_tokens = MAX_NEW_TOKENS,
        temperature    = temperature,
        do_sample      = temperature > 0,
        pad_token_id   = llm_tok.eos_token_id,
    )
    decoded = llm_tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return parse_scores(decoded)


def rag_predict(row: pd.Series) -> List[str]:
    """
    Full pipeline:
      1. Multi-query hybrid retrieval (3 variants)
      2. BGE-large rerank → top-6
      3. Self-consistency: 2 Qwen passes at different temperatures
      4. Average scores → top-3 choices
    """
    queries  = expand_queries(row)
    evidence = retrieve(queries, top_k=RERANK_TOP_K)
    user_msg = build_prompt(row, evidence)

    # Self-consistency: multiple temperatures, average
    all_scores = [
        single_pass(user_msg, temp, seed=42 + i)
        for i, temp in enumerate(TEMPERATURES[:SELF_CONSIST_N])
    ]

    # Average across passes
    avg = {c: np.mean([s.get(c, 0.0) for s in all_scores]) for c in CHOICES}

    return sorted(CHOICES, key=lambda c: avg[c], reverse=True)[:3]


print("✅ rag_predict() ready (self-consistency enabled)")

In [ ]:
# ── Smoke Test ────────────────────────────────────────────────────────────────
print("Running smoke test on 1 row...")
row = test_df.iloc[0]
print("Q:", row["prompt"][:100])
pred = rag_predict(row)
print("Prediction:", pred)

In [ ]:
# ── Validation MAP@3 (out-of-fold, NO data leakage) ──────────────────────────
# The val rows are excluded from the KB build above.
# This gives a REAL estimate of test performance.

def ap3(pred, truth):
    for i, p in enumerate(pred[:3], 1):
        if p == truth: return 1.0/i
    return 0.0

EVAL_N   = 100
val_df   = train_df.sample(n=EVAL_N, random_state=42).reset_index(drop=True)
aps      = []

for _, row in tqdm(val_df.iterrows(), total=EVAL_N, desc="Validation"):
    pred = rag_predict(row)
    aps.append(ap3(pred, row["answer"]))

print(f"\n=== Validation Results ({EVAL_N} samples) ===")
print(f"MAP@3     : {np.mean(aps):.4f}")
print(f"Top-1 Acc : {np.mean([a==1.0  for a in aps]):.4f}")
print(f"Top-2 Acc : {np.mean([a>=0.5  for a in aps]):.4f}")
print(f"Top-3 Acc : {np.mean([a >0.0  for a in aps]):.4f}")

In [ ]:
# ── Run on All 500 Test Questions ─────────────────────────────────────────────
results = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test inference"):
    top3 = rag_predict(row)
    results.append({"ID": row["id"], "Prediction": " ".join(top3)})

print(f"\n✅ {len(results)} predictions complete")

In [ ]:
# ── Save submission.csv ───────────────────────────────────────────────────────
sub      = pd.DataFrame(results)
sub_path = OUT_DIR / "submission.csv"
sub.to_csv(sub_path, index=False)

# Format checks
assert list(sub.columns) == ["ID", "Prediction"],                  "Wrong columns"
assert len(sub) == len(test_df),                                    "Wrong row count"
assert sub["Prediction"].str.match(r'^[A-E] [A-E] [A-E]$').all(), "Wrong prediction format"

print(f"✅ Saved → {sub_path}")
print(f"Preview:")
print(sub.head(10).to_string(index=False))